In [1]:
# ==============================================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS (Sem conectores Java do S3)
# ==============================================================================
!pip install pyspark boto3 pandas -q

from google.colab import userdata
userdata.get('secretName')
import os
import boto3
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# ==============================================================================
# 2. INICIALIZAÇÃO DO SPARK (Simples e Leve)
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Ingestao-Colab-Bronze") \
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
    .getOrCreate()

# ==============================================================================
# 3. CONFIGURAÇÃO DO CLIENTE AWS NATIVO (Boto3)
# ==============================================================================
AWS_ACCESS_KEY = "AWS_KEY"
AWS_SECRET_KEY = "AWS_SECRET"
AWS_SESSION_TOKEN = "SESSION_TOKEN"
# Cria o cliente S3 nativo do Python (não passa pelo Java/Hadoop)
if AWS_SESSION_TOKEN and AWS_SESSION_TOKEN != "SESSION_TOKEN":
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        aws_session_token=AWS_SESSION_TOKEN
    )
else:
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY
    )

# ==============================================================================
# 4. VARIÁVEIS GLOBAIS DE TEMPO E BUCKETS
# ==============================================================================
INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
ANOMESDIA = datetime.now(timezone.utc).strftime("%Y%m%d")

BUCKET_PRINICIPAL = "fiap-datalake-fase2"
PASTA_RAW = "raw"
PASTA_BRONZE = "bronze"

print("✅ Ambiente Spark + Boto3 configurado com sucesso!")

✅ Ambiente Spark + Boto3 configurado com sucesso!


In [2]:
# ==============================================================================
# CÉLULA 2: MAPEAMENTO DE SCHEMAS E ARQUIVOS REAIS DO S3
# ==============================================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Seus nomes de entidades organizadas (como vão ficar as pastas na Bronze)
ENTIDADES = [
    "meta_brasil", "meta_uf", "meta_municipio",
    "municipio", "uf", "alunos_2023", "alunos_2024", "alunos_2025"
]

# DICIONÁRIO DE DEPARA: Relaciona a entidade com o nome REAL do arquivo no S3
ARQUIVOS_MAP = {
    "meta_brasil": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv",
    "meta_uf": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
    "meta_municipio": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv",
    "municipio": "br_inep_avaliacao_alfabetizacao_municipio.csv",
    "uf": "br_inep_avaliacao_alfabetizacao_uf.csv",
    "alunos_2023": "TS_ALUNO_2023.csv",
    "alunos_2024": "TS_ALUNO_2024.csv",
    "alunos_2025": "TS_ALUNO_2025.csv"
}

# Dicionário de schemas
SCHEMAS_MAP = {
    "meta_brasil": StructType([
        StructField("ano", IntegerType(), True),
        StructField("rede", StringType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("meta_alfabetizacao_2024", DoubleType(), True),
        StructField("meta_alfabetizacao_2025", DoubleType(), True),
        StructField("meta_alfabetizacao_2026", DoubleType(), True),
        StructField("meta_alfabetizacao_2027", DoubleType(), True),
        StructField("meta_alfabetizacao_2028", DoubleType(), True),
        StructField("meta_alfabetizacao_2029", DoubleType(), True),
        StructField("meta_alfabetizacao_2030", DoubleType(), True),
        StructField("percentual_participacao", DoubleType(), True)
    ]),
    "meta_uf": StructType([
        StructField("ano", IntegerType(), True),
        StructField("sigla_uf", StringType(), True),
        StructField("rede", StringType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("meta_alfabetizacao_2024", DoubleType(), True),
        StructField("meta_alfabetizacao_2025", DoubleType(), True),
        StructField("meta_alfabetizacao_2026", DoubleType(), True),
        StructField("meta_alfabetizacao_2027", DoubleType(), True),
        StructField("meta_alfabetizacao_2028", DoubleType(), True),
        StructField("meta_alfabetizacao_2029", DoubleType(), True),
        StructField("meta_alfabetizacao_2030", DoubleType(), True),
        StructField("percentual_participacao", DoubleType(), True)
    ]),
    "meta_municipio": StructType([
        StructField("ano", IntegerType(), True),
        StructField("id_municipio", IntegerType(), True),
        StructField("rede", StringType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("meta_alfabetizacao_2024", DoubleType(), True),
        StructField("meta_alfabetizacao_2025", DoubleType(), True),
        StructField("meta_alfabetizacao_2026", DoubleType(), True),
        StructField("meta_alfabetizacao_2027", DoubleType(), True),
        StructField("meta_alfabetizacao_2028", DoubleType(), True),
        StructField("meta_alfabetizacao_2029", DoubleType(), True),
        StructField("meta_alfabetizacao_2030", DoubleType(), True),
        StructField("nivel_alfabetizacao", DoubleType(), True),
        StructField("percentual_participacao", DoubleType(), True)
    ]),
    "municipio": StructType([
        StructField("ano", IntegerType(), True),
        StructField("id_municipio", IntegerType(), True),
        StructField("serie", IntegerType(), True),
        StructField("rede", IntegerType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("media_portugues", DoubleType(), True),
        StructField("proporcao_aluno_nivel_0", DoubleType(), True),
        StructField("proporcao_aluno_nivel_1", DoubleType(), True),
        StructField("proporcao_aluno_nivel_2", DoubleType(), True),
        StructField("proporcao_aluno_nivel_3", DoubleType(), True),
        StructField("proporcao_aluno_nivel_4", DoubleType(), True),
        StructField("proporcao_aluno_nivel_5", DoubleType(), True),
        StructField("proporcao_aluno_nivel_6", DoubleType(), True),
        StructField("proporcao_aluno_nivel_7", DoubleType(), True),
        StructField("proporcao_aluno_nivel_8", DoubleType(), True)
    ]),
    "uf": StructType([
          StructField("ano", IntegerType(), True),
          StructField("sigla_uf", StringType(), True),
          StructField("serie", IntegerType(), True),
          StructField("rede", IntegerType(), True),
          StructField("taxa_alfabetizacao", DoubleType(), True),
          StructField("media_portugues", DoubleType(), True),
          StructField("proporcao_aluno_nivel_0", DoubleType(), True),
          StructField("proporcao_aluno_nivel_1", DoubleType(), True),
          StructField("proporcao_aluno_nivel_2", DoubleType(), True),
          StructField("proporcao_aluno_nivel_3", DoubleType(), True),
          StructField("proporcao_aluno_nivel_4", DoubleType(), True),
          StructField("proporcao_aluno_nivel_5", DoubleType(), True),
          StructField("proporcao_aluno_nivel_6", DoubleType(), True),
          StructField("proporcao_aluno_nivel_7", DoubleType(), True),
          StructField("proporcao_aluno_nivel_8", DoubleType(), True)
    ]),
    "alunos_2023": StructType([
        StructField("NU_ANO_AVALIACAO", IntegerType(), True),
        StructField("CO_UF", IntegerType(), True),
        StructField("SG_UF", StringType(), True),
        StructField("ID_ALUNO", IntegerType(), True),
        StructField("TP_SERIE", IntegerType(), True),
        StructField("ID_ESCOLA", IntegerType(), True),
        StructField("TP_DEPENDENCIA", IntegerType(), True),
        StructField("CO_MUNICIPIO", IntegerType(), True),
        StructField("NO_MUNICIPIO", StringType(), True),
        StructField("IN_PRESENCA_LP", IntegerType(), True),
        StructField("IN_PREENCHIMENTO_LP", IntegerType(), True),
        StructField("CO_CADERNO_LP", IntegerType(), True),
        StructField("VL_PESO_ALUNO_LP", DoubleType(), True),
        StructField("VL_PROFICIENCIA_LP", DoubleType(), True),
        StructField("IN_ALFABETIZADO", IntegerType(), True)
    ]),
    "alunos_2024": StructType([
        StructField("NU_ANO_AVALIACAO", IntegerType(), True),
        StructField("CO_UF", IntegerType(), True),
        StructField("SG_UF", StringType(), True),
        StructField("ID_ALUNO", IntegerType(), True),
        StructField("TP_SERIE", IntegerType(), True),
        StructField("ID_ESCOLA", IntegerType(), True),
        StructField("TP_DEPENDENCIA", IntegerType(), True),
        StructField("CO_MUNICIPIO", IntegerType(), True),
        StructField("NO_MUNICIPIO", StringType(), True),
        StructField("IN_PRESENCA_LP", IntegerType(), True),
        StructField("IN_PREENCHIMENTO_LP", IntegerType(), True),
        StructField("CO_CADERNO_LP", IntegerType(), True),
        StructField("VL_PESO_ALUNO_LP", DoubleType(), True),
        StructField("VL_PROFICIENCIA_LP", DoubleType(), True),
        StructField("IN_ALFABETIZADO", IntegerType(), True)
    ]),
    "alunos_2025": StructType([
        StructField("NU_ANO_AVALIACAO", IntegerType(), True),
        StructField("CO_UF", IntegerType(), True),
        StructField("SG_UF", StringType(), True),
        StructField("ID_ALUNO", IntegerType(), True),
        StructField("TP_SERIE", IntegerType(), True),
        StructField("ID_ESCOLA", IntegerType(), True),
        StructField("TP_DEPENDENCIA", IntegerType(), True),
        StructField("CO_MUNICIPIO", IntegerType(), True),
        StructField("NO_MUNICIPIO", StringType(), True),
        StructField("IN_PRESENCA_LP", IntegerType(), True),
        StructField("IN_PREENCHIMENTO_LP", IntegerType(), True),
        StructField("CO_CADERNO_LP", IntegerType(), True),
        StructField("VL_PESO_ALUNO_LP", DoubleType(), True),
        StructField("VL_PROFICIENCIA_LP", DoubleType(), True),
        StructField("IN_ALFABETIZADO", IntegerType(), True)
    ]),

}
print("✅ Mapeamento de arquivos físicos carregado com sucesso!")

✅ Mapeamento de arquivos físicos carregado com sucesso!


In [9]:
def fazer_upload_pasta_s3(pasta_local, bucket, prefixo_s3):
    """
    Função auxiliar para varrer a pasta local do Colab e enviar os arquivos ao S3
    """
    import os
    for raiz, diretorios, arquivos in os.walk(pasta_local):
        for arquivo in arquivos:
            caminho_completo_local = os.path.join(raiz, arquivo)
            # Calcula o caminho relativo para manter a estrutura de pastas no S3
            caminho_relativo = os.path.relpath(caminho_completo_local, pasta_local)
            key_s3 = os.path.join(prefixo_s3, caminho_relativo).replace("\\", "/")

            # Envia o arquivo para o S3 via Boto3
            s3_client.upload_file(caminho_completo_local, bucket, key_s3)

def executar_pipeline_bronze(entidade, bucket, pasta_in, pasta_out):
    """
    Lê o CSV do S3 tratando encoding e identificando o separador (, ou ;) de forma
    automática, aplica tipagem via .cast() tratando strings 'NaN'/nulas, adiciona metadados,
    grava em Parquet localmente e faz o upload dos arquivos gerados de volta para o S3.
    """
    print(f"\n--- Iniciando processamento da entidade: {entidade} ---")

    # Busca o nome real do arquivo no dicionário mapeado
    nome_arquivo_real = ARQUIVOS_MAP.get(entidade, f"{entidade}.csv")

    key_input_csv = f"{pasta_in}/{nome_arquivo_real}"

    # Define a estrutura de partições
    estrutura_destino = f"{entidade}/anomesdia={ANOMESDIA}"
    path_local_output = f"./output/{pasta_out}/{estrutura_destino}"
    key_s3_output = f"{pasta_out}/{estrutura_destino}"

    print(f"[RAW] Baixando objeto do S3 via Boto3: {key_input_csv}")

    # 1. Leitura do S3 tratando dinamicamente Encoding e Separador
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key_input_csv)
        corpo_arquivo = obj['Body'].read()

        try:
            # Tenta UTF-8 descobrindo o separador automaticamente
            pdf = pd.read_csv(pd.io.common.BytesIO(corpo_arquivo), sep=None, engine='python', encoding="utf-8", dtype=str)
        except UnicodeDecodeError:
            print(f"[AVISO] Falha ao decodificar em UTF-8 para '{entidade}'. Tentando padrão ISO-8859-1 (Latin1)...")
            # Fallback para Latin1 descobrindo o separador automaticamente
            pdf = pd.read_csv(pd.io.common.BytesIO(corpo_arquivo), sep=None, engine='python', encoding="ISO-8859-1", dtype=str)

    except Exception as e:
        print(f"[ERRO] Falha ao ler ou decodificar arquivo do S3 via Boto3: {str(e)}")
        raise e

    # 2. Captura o schema específico se existir, caso contrário lê genérico
    schema_especifico = SCHEMAS_MAP.get(entidade, None)

    # Convertendo o DataFrame do Pandas para Spark DataFrame
    if schema_especifico:
        print(f"[RAW] Schema explícito encontrado para '{entidade}'. Ajustando tipos e tratando NaNs...")

        # Cria o DataFrame temporário flexível
        df_temp = spark.createDataFrame(pdf)

        # Intercepta strings textuais nulas/NaN antes de realizar o .cast()
        expressoes_tipo = [
            F.when(F.col(campo.name).isin(["NaN", "nan", "None", "NULL", "nan"]), F.lit(None))
             .otherwise(F.col(campo.name))
             .cast(campo.dataType)
             .alias(campo.name)
            for campo in schema_especifico
        ]
        df_raw = df_temp.select(*expressoes_tipo)
    else:
        print(f"[RAW] Nenhum schema mapeado para '{entidade}'. Convertendo como texto.")
        df_raw = spark.createDataFrame(pdf)

    # 3. Adição de Metadados de Auditoria
    print("[BRONZE] Aplicando metadados de auditoria...")
    path_fake_s3 = f"s3://{bucket}/{key_input_csv}"
    df_bronze = (df_raw
        .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
        .withColumn("_ingestion_date",      F.lit(INGESTION_DATE))
        .withColumn("_source_path",         F.lit(path_fake_s3))
        .withColumn("_source_entity",       F.lit(entidade))
        .withColumn("_environment",         F.lit("dev"))
    )

    # 4. Escrita no formato Parquet (Local no ambiente do Colab)
    print(f"[BRONZE] Salvando Parquet temporariamente no Colab...")
    df_bronze.write.mode("overwrite").parquet(path_local_output)

    # 5. Upload da estrutura Parquet gerada de volta para o S3
    print(f"[S3-UPLOAD] Enviando arquivos Parquet para o S3 em: s3://{bucket}/{key_s3_output}/")
    try:
        fazer_upload_pasta_s3(path_local_output, bucket, key_s3_output)
        print(f"[S3-UPLOAD] Upload concluído com sucesso!")
    except Exception as e:
        print(f"[ERRO] Falha ao enviar arquivos para o S3: {str(e)}")
        raise e

    print(f"[SUCESSO] Entidade '{entidade}' processada e salva no S3. Registros: {df_bronze.count()}")

In [10]:
# Execução em loop para as tabelas
for entidade in ENTIDADES:
    try:
        executar_pipeline_bronze(
            entidade=entidade,
            bucket=BUCKET_PRINICIPAL,
            pasta_in=PASTA_RAW,
            pasta_out=PASTA_BRONZE
        )
    except Exception as e:
        print(f"[ERRO] Falha ao processar a entidade {entidade}: {str(e)}")


--- Iniciando processamento da entidade: meta_brasil ---
[RAW] Baixando objeto do S3 via Boto3: raw/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv
[RAW] Schema explícito encontrado para 'meta_brasil'. Ajustando tipos e tratando NaNs...
[BRONZE] Aplicando metadados de auditoria...
[BRONZE] Salvando Parquet temporariamente no Colab...
[S3-UPLOAD] Enviando arquivos Parquet para o S3 em: s3://fiap-datalake-fase2/bronze/meta_brasil/anomesdia=20260623/
[S3-UPLOAD] Upload concluído com sucesso!
[SUCESSO] Entidade 'meta_brasil' processada e salva no S3. Registros: 3

--- Iniciando processamento da entidade: meta_uf ---
[RAW] Baixando objeto do S3 via Boto3: raw/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv
[RAW] Schema explícito encontrado para 'meta_uf'. Ajustando tipos e tratando NaNs...
[BRONZE] Aplicando metadados de auditoria...
[BRONZE] Salvando Parquet temporariamente no Colab...
[S3-UPLOAD] Enviando arquivos Parquet para o S3 em: s3://fiap-datalake-fase2/bro